# Reflection from plane and curved mirrors

*optics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/mirror_reflection)

The law of reflection written as one vector formula: flip the part of the ray direction that points along the surface normal. Rays from a lamp bounce off a flat mirror, and parallel rays bounce off a concave mirror and gather near its focal point $F = R/2$.

**Vector form**

$$\mathbf r = \mathbf d - 2\,(\mathbf d\cdot\mathbf n)\,\mathbf n$$

**Index (tensor) form**

$$r_{ri} = d_{ri} - 2\,(d_{rj}\,n_{rj})\,n_{ri}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $d_{ri}$ | `d` | (R, 2) | unit direction of incoming ray $r$, component $i \in \{x, y\}$ |
| $n_{ri}$ | `normal` | (R, 2) | unit normal of the mirror where ray $r$ hits it |
| $d_{rj}n_{rj}$ | `einsum('rj,rj->r', d, n)` | (R,) | repeated $j$ ⇒ summed: one dot product per ray (${}= -\cos\theta_i$) |
| $r_{ri}$ | `reflect(d, n)` | (R, 2) | reflected direction, still a unit vector |
| $n_{ri} = (c_i - p_{ri})/R$ | `sphere_normal(p_hit, centre, R)` | (R, 2) | normal of the curved mirror points to its centre $c$ |

The formula only flips the component of $\mathbf d$ along $\mathbf n$, so the angle of incidence equals the angle of reflection automatically. Dashed lines continue the reflected rays behind the flat mirror: they all meet at the virtual image of the lamp. For the concave mirror only rays close to the axis (paraxial rays) pass through $F = R/2$; rays further out cross the axis closer to the mirror (spherical aberration).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/mirror_reflection/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    radius, n_rays, aperture = params["radius"], params["n_rays"], params["aperture"]
    lamp = tensors.lamp_rays(radius, n_rays, aperture)
    parallel = tensors.parallel_rays(radius, n_rays, aperture)
    flat = tensors.flat_mirror()
    centre = tensors.mirror_centre(radius)
    times = tensors.times(radius)

    result = solve.solve(radius, lamp, parallel, flat, centre, times)

    print(f"focal point F = R/2 at x = {-radius / 2:.2f} m")
    for y, x in zip(parallel[0][:, 1].tolist(), result["crossings"].tolist()):
        if y > 0:
            print(f"ray at height {y:5.2f} m crosses the axis at x = {x:6.3f} m")
    return plot.animate(times, result, radius, centre, out_dir / "mirror_reflection.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/mirror_reflection/equation.py
r"""Physics → tensor translation.

The law of reflection (angle in = angle out) in vector form: for a unit ray direction \mathbf d
hitting a mirror with unit normal \mathbf n,

    \mathbf r = \mathbf d - 2 (\mathbf d\cdot\mathbf n)\, \mathbf n

The dot product picks out the part of d along the normal; subtracting it twice flips that part.
In index notation for a batch of rays r and spatial components i:

    r_{ri} = d_{ri} - 2 (d_{rj} n_{rj})\, n_{ri}        (repeated j ⇒ summed)

For a spherical mirror with centre c_i and radius R the normal at the hit point p_{ri} is

    n_{ri} = (c_i - p_{ri}) / R
"""

import torch


def dot(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    r"""a_{rj} b_{rj}: one dot product per ray.   (R, 2), (R, 2) → (R,)"""
    return torch.einsum("rj,rj->r", a, b)


def reflect(d: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""r_{ri} = d_{ri} - 2 (d_{rj} n_{rj}) n_{ri}.   (R, 2) → (R, 2)"""
    return d - 2 * dot(d, n)[:, None] * n


def sphere_normal(p: torch.Tensor, centre: torch.Tensor, radius: float) -> torch.Tensor:
    r"""n_{ri} = (c_i - p_{ri}) / R: unit normal of a sphere, pointing to its centre.   (R, 2) → (R, 2)"""
    return (centre - p) / radius


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/mirror_reflection/tensors.py
"""Build the concrete tensors: the lamp's fan of rays for the flat mirror, parallel rays for the curved one.

Both mirrors sit at x = 0 and the light arrives from the left (x < 0).
"""

import torch


def lamp_rays(radius: float, n_rays: int, aperture: float) -> tuple[torch.Tensor, torch.Tensor]:
    """A lamp at (−R/2, R/4) shines at points spread along the flat mirror x = 0.

    p[r, :] = lamp position                 start point p_{ri}       (R, 2)
    d[r, :] = unit vector lamp → mirror     direction d_{ri}         (R, 2)
    """
    lamp = torch.tensor([-0.5 * radius, 0.25 * radius])  # (2,)
    y_target = torch.linspace(-aperture, aperture, n_rays) * radius  # (R,) where each ray aims
    target = torch.stack([torch.zeros_like(y_target), y_target], dim=-1)  # (R, 2)
    d = target - lamp
    d = d / d.norm(dim=-1, keepdim=True)  # make every direction a unit vector
    return lamp.expand_as(d), d


def parallel_rays(radius: float, n_rays: int, aperture: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Rays parallel to the axis, at heights between −aperture·R and +aperture·R, starting at x = −1.2 R.

    p[r, :] = (−1.2 R, y_r)        (R, 2)
    d[r, :] = (1, 0)               (R, 2)
    """
    y = torch.linspace(-aperture, aperture, n_rays) * radius  # (R,)
    p = torch.stack([torch.full_like(y, -1.2 * radius), y], dim=-1)
    d = torch.tensor([1.0, 0.0]).expand_as(p)
    return p, d


def flat_mirror() -> tuple[torch.Tensor, torch.Tensor]:
    """A point on the flat mirror and its unit normal (facing the light): two (2,) tensors."""
    return torch.tensor([0.0, 0.0]), torch.tensor([-1.0, 0.0])


def mirror_centre(radius: float) -> torch.Tensor:
    """Centre of curvature of the concave mirror whose vertex is at the origin: (2,) = (−R, 0)."""
    return torch.tensor([-radius, 0.0])


def times(radius: float, n_steps: int = 120) -> torch.Tensor:
    """Time samples (light speed 1), long enough for rays to come back past the focus: (T,)."""
    return torch.linspace(0.0, 2.2 * radius, n_steps)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/mirror_reflection/solve.py
"""Find where each ray meets its mirror, reflect it, and record the ray tip at every time."""

import torch

from .equation import dot, reflect, sphere_normal


def hit_flat(p: torch.Tensor, d: torch.Tensor, point: torch.Tensor, normal: torch.Tensor) -> torch.Tensor:
    """Distance t along each ray to the plane through `point` with unit `normal`: (R,).

    Solve (p + t d − a)·n = 0  for t.
    """
    n = normal.expand_as(d)
    return dot(point - p, n) / dot(d, n)


def hit_sphere(p: torch.Tensor, d: torch.Tensor, centre: torch.Tensor, radius: float) -> torch.Tensor:
    """Distance t along each ray to the sphere |p + t d − c| = R: (R,).

    With q = p − c this is the quadratic t² + 2 (q·d) t + (q·q − R²) = 0. The rays start inside
    the sphere, so the + root is the one in front of them (the mirror on the right).
    """
    q = p - centre  # (R, 2)
    b = dot(q, d)  # (R,)
    c = dot(q, q) - radius**2  # (R,)
    return -b + torch.sqrt(b**2 - c)


def bounce(p: torch.Tensor, d: torch.Tensor, t_hit: torch.Tensor, normal: torch.Tensor):
    """Return (hit points (R, 2), reflected directions (R, 2))."""
    p_hit = p + t_hit[:, None] * d
    return p_hit, reflect(d, normal)


def tips(times: torch.Tensor, p, d, t_hit, p_hit, r) -> torch.Tensor:
    """Position of every ray tip at every time: (T,) → (T, R, 2).

    Before the bounce the tip is at p + t d; afterwards at p_hit + (t − t_hit) r.
    """
    t = times[:, None, None]  # (T, 1, 1) so it broadcasts against (R, 2)
    before = p + t * d
    after = p_hit + (t - t_hit[:, None]) * r
    return torch.where(t < t_hit[:, None], before, after)


def axis_crossing(p_hit: torch.Tensor, r: torch.Tensor) -> torch.Tensor:
    """x where each reflected ray crosses the axis y = 0: (R,).

    Walk back along r from the hit point until y = 0:  x = p_x − p_y r_x / r_y.
    (A ray travelling exactly on the axis gives 0/0 = NaN.)
    """
    return p_hit[:, 0] - p_hit[:, 1] * r[:, 0] / r[:, 1]


def solve(radius: float, lamp: tuple, parallel: tuple, flat: tuple, centre: torch.Tensor, times: torch.Tensor):
    """Trace both scenes. Returns a dict of tensors for the plot."""
    p1, d1 = lamp
    t1 = hit_flat(p1, d1, *flat)
    hit1, r1 = bounce(p1, d1, t1, flat[1].expand_as(d1))

    p2, d2 = parallel
    t2 = hit_sphere(p2, d2, centre, radius)
    hit2 = p2 + t2[:, None] * d2
    hit2, r2 = bounce(p2, d2, t2, sphere_normal(hit2, centre, radius))

    return {
        "flat": tips(times, p1, d1, t1, hit1, r1),  # (T, R, 2) ray tips
        "flat_start": p1,  # (R, 2)
        "flat_hits": hit1,  # (R, 2)
        "flat_t_hit": t1,  # (R,)
        "flat_dirs": r1,  # (R, 2)
        "curved": tips(times, p2, d2, t2, hit2, r2),  # (T, R, 2)
        "curved_start": p2,
        "curved_hits": hit2,
        "curved_t_hit": t2,
        "crossings": axis_crossing(hit2, r2),  # (R,)
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/mirror_reflection/plot.py
"""Animate light rays travelling to the flat mirror (left) and the concave mirror (right) and bouncing back."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(times: torch.Tensor, res: dict, radius: float, centre: torch.Tensor, path: Path) -> Path:
    t = times.numpy()
    R = radius
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 4))
    for ax in (ax1, ax2):
        ax.set_xlim(-1.25 * R, 0.6 * R)
        ax.set_ylim(-1.0 * R, 1.0 * R)
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
    ax1.set_ylabel("y (m)")

    # flat mirror, lamp and its virtual image (mirror image of the lamp behind x = 0)
    ax1.axvline(0, color="#374151", lw=3)
    ax1.plot([-0.5 * R], [0.25 * R], "o", color="#ea580c", ms=8, label="lamp")
    ax1.plot([0.5 * R], [0.25 * R], "o", mfc="none", color="#ea580c", ms=8, label="virtual image")
    ax1.legend(loc="lower left", fontsize=7)
    ax1.set_title("flat mirror")

    # concave mirror: arc of the circle |p − c| = R near the vertex, plus C and F on the axis
    c = centre.numpy()
    phi = np.linspace(-1.1, 1.1, 100)
    ax2.plot(c[0] + R * np.cos(phi), c[1] + R * np.sin(phi), color="#374151", lw=3)
    ax2.axhline(0, color="#9ca3af", lw=0.8)
    ax2.plot([-R], [0], "k+", ms=10)
    ax2.text(-R, -0.9 * R, "C", ha="center", va="bottom")
    ax2.plot([-R / 2], [0], "k*", ms=10, zorder=5)
    ax2.text(-R / 2, -0.9 * R, "F = R/2", ha="center", va="bottom")
    ax2.vlines([-R, -R / 2], -0.8 * R, 0, colors="#9ca3af", lw=0.8, ls=":")
    ax2.set_title("concave mirror")

    def make_lines(ax, n):
        colors = plt.get_cmap("plasma")(np.linspace(0, 0.85, n))
        return [ax.plot([], [], lw=1.2, color=col)[0] for col in colors]

    n_rays = res["flat"].shape[1]
    flat_lines, curved_lines = make_lines(ax1, n_rays), make_lines(ax2, n_rays)
    ghosts = [ax1.plot([], [], ":", lw=0.8, color=line.get_color())[0] for line in flat_lines]

    def draw(lines, start, hits, t_hit, tip, now):  # start, hits: (R, 2), t_hit: (R,), tip: (R, 2)
        """Each ray is the polyline start → (hit point, once it has bounced) → current tip."""
        for r, line in enumerate(lines):
            if now < t_hit[r]:
                pts = np.array([start[r], tip[r]])
            else:
                pts = np.array([start[r], hits[r], tip[r]])
            line.set_data(pts[:, 0], pts[:, 1])

    flat_start, flat_hits, flat_t_hit, flat_tips = (
        res[k].numpy() for k in ("flat_start", "flat_hits", "flat_t_hit", "flat")
    )
    curved_start, curved_hits, curved_t_hit, curved_tips = (
        res[k].numpy() for k in ("curved_start", "curved_hits", "curved_t_hit", "curved")
    )
    image = np.array([0.5 * R, 0.25 * R])

    def update(step: int):
        now = t[step]
        draw(flat_lines, flat_start, flat_hits, flat_t_hit, flat_tips[step], now)
        draw(curved_lines, curved_start, curved_hits, curved_t_hit, curved_tips[step], now)
        for r, ghost in enumerate(ghosts):  # extend bounced rays backwards: they meet at the virtual image
            if now >= flat_t_hit[r]:
                ghost.set_data([flat_hits[r, 0], image[0]], [flat_hits[r, 1], image[1]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path, fps=15)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'radius': 4.0,
    'n_rays': 9,
    'aperture': 0.6
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.mirror_reflection")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.mirror_reflection.main")
problem_dir = Path("problems/optics/mirror_reflection")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
